# Quickstart

In [ ]:
import warnings

import plotnine as p9

import evy

# Ignore casting warning from xarray/numpy
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    message=r"invalid value encountered in cast",
)

## 1. Loading EVI Data with Quality Masking

The `open_evi` function loads MODIS EVI data and can apply quality masking to filter out quality pixels based on VI quality and pixel reliability flags.

In [ ]:
adm0 = evy.fetch_boundaries("SMR", 0, "gbOpen", output_dir="../data/")
adm1 = evy.fetch_boundaries("SMR", 1, "gbOpen", output_dir="../data/")
bbox = adm0.geometry[0].bounds
date_range = ("2020-01-01", "2023-12-31")

evi_data = evy.open_evi(
    product="MOD13Q1",
    bbox=bbox,
    date_range=date_range,
    mask_quality=True,
    confidence_threshold=0,
)

## 2. Calculate Zonal Statistics

Compute statistics (mean, median, max, etc.) for specific geometries. You can also specify temporal aggregation frequencies.

In [ ]:
zonal_monthly = evy.zonal_stats(
    evi_data,
    geometries=adm1,
    stats_funcs="mean",
    temporal_agg="ME",
)

zonal_monthly.head()

In [ ]:
zonal_annual = evy.zonal_stats(
    evi_data,
    geometries=adm1,
    stats_funcs="mean",
    temporal_agg="YE",
)

zonal_annual.query("time > '2023-01-01'").plot(column="evi", legend=True, cmap="Greens")

## 3. Plot Aggregated EVI Data

Visualize the aggregated EVI data with various temporal faceting options (monthly, yearly).

In [ ]:
monthly_plot_data = zonal_monthly.copy().assign(
    year=lambda df: df["time"].dt.year, month=lambda df: df["time"].dt.month
)
ts_plot = (
    p9.ggplot(monthly_plot_data)
    + p9.geom_line(p9.aes(x="time", y="evi", color="shapeName"), size=1)
    + p9.geom_point(p9.aes(x="time", y="evi", color="shapeName"), size=0.8, alpha=0.7)
    + p9.theme_minimal()
    + p9.labs(
        title="Monthly EVI Time Series by Region",
        x="Time",
        y="Mean EVI",
        color="Region",
    )
    + p9.theme(axis_text_x=p9.element_text(rotation=45))
)
ts_plot

In [ ]:
facet_year_plot = (
    p9.ggplot(monthly_plot_data)
    + p9.geom_line(p9.aes(x="month", y="evi", color="shapeName"), size=1)
    + p9.geom_point(p9.aes(x="month", y="evi", color="shapeName"), size=0.8)
    + p9.facet_wrap("year", ncol=2)
    + p9.theme_minimal()
    + p9.labs(
        title="Monthly EVI by Region (Faceted by Year)",
        x="Month",
        y="Mean EVI",
        color="Region",
    )
    + p9.scale_x_continuous(breaks=range(1, 13))
)
facet_year_plot

## 4. Plot Phenology

Analyze and visualize vegetation growing seasons and phenological patterns.

In [ ]:
phenology_annual = evy.compute_phenology(
    evi_data,
    geometries=adm1,
    stats_funcs="mean",
    aggregate_years=False,
)
phenology_annual.head()

Compute the phenology metrics for each year and each region.

In [ ]:
evy.plot_phenology(
    evi_data,
    geometries=adm1,
    aggregate_years=False,
)

Compute the phenology metrics for each region across all years.

In [ ]:
evy.plot_phenology(
    evi_data,
    geometries=adm1,
    aggregate_years=True,
)

## 5. Calculate Anomalies

Calculate vegetation anomalies by comparing target periods against baseline. Returns raw z-scores without spatial or temporal aggregation.

In [ ]:
baseline_years = (2020, 2021)
target_years = (2022, 2023)

anomalies_zscore = evy.compute_anomalies(
    evi_data, baseline_years=baseline_years, target_years=target_years, method="zscore"
)

anomalies_diff = evy.compute_anomalies(
    evi_data,
    baseline_years=baseline_years,
    target_years=target_years,
    method="difference",
)

anomalies_pct = evy.compute_anomalies(
    evi_data,
    baseline_years=baseline_years,
    target_years=target_years,
    method="percentage",
)

## 6. Aggregated Anomalies

Use convenient function to chain the anomaly calculation and zonal statistics to get aggregated anomaly metrics for specific geometries. 

In [ ]:
aggregated_yearly_anomalies = evy.compute_aggregated_anomalies(
    evi_data,
    geometries=adm1,
    baseline_years=baseline_years,
    target_years=target_years,
    method="zscore",
    stats_funcs=["mean", "std"],
    temporal_freq="YE",
)


anomaly_ts_plot = (
    p9.ggplot(aggregated_yearly_anomalies)
    + p9.geom_line(p9.aes(x="time", y="evi", color="shapeName"), size=1)
    + p9.geom_point(p9.aes(x="time", y="evi", color="shapeName"), size=0.8)
    + p9.geom_hline(yintercept=0, linetype="dashed", color="gray")
    + p9.geom_hline(yintercept=[-1, 1], linetype="dotted", color="orange", alpha=0.7)
    + p9.geom_hline(yintercept=[-2, 2], linetype="dotted", color="red", alpha=0.7)
    + p9.theme_minimal()
    + p9.labs(
        title="EVI Z-score Anomalies by Region",
        subtitle="Target: 2022-2023, Baseline: 2020-2021",
        x="Time",
        y="Mean Z-score Anomaly",
        color="Region",
    )
    + p9.theme(axis_text_x=p9.element_text(rotation=45))
)
anomaly_ts_plot

In [ ]:
aggregated_yearly_anomalies.plot(column="evi", cmap="coolwarm", legend=True)

In [ ]:
(
    p9.ggplot(aggregated_yearly_anomalies)
    + p9.geom_map(p9.aes(fill="evi"))
    + p9.facet_wrap("time")
    + p9.scale_fill_continuous(cmap_name="coolwarm")
    + p9.coord_fixed(expand=False)
    + p9.theme_void()
)